In [ ]:
!nvidia-smi

Tue Aug  4 13:56:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import libraries
import pandas as pd
import numpy as np

print("=" * 60)
print("IMPORTING DATA FROM GOOGLE DRIVE")
print("=" * 60)

# Correct path with space
data_path = '/content/drive/My Drive/Data'

# Load all datasets with correct filenames
print("\n📥 Loading datasets...")

train_df = pd.read_csv(f'{data_path}/train-test.csv')
validation_df = pd.read_csv(f'{data_path}/validation.csv')
template_df = pd.read_csv(f'{data_path}/validation-predictions-template.csv')
december_df = pd.read_csv(f'{data_path}/december-chart-inputs (1).csv')

print("✅ All files loaded successfully!\n")

# Display data information
print("📊 DATASET SUMMARY:")
print("-" * 60)
print(f"Train data shape: {train_df.shape}")
print(f"Validation data shape: {validation_df.shape}")
print(f"Template data shape: {template_df.shape}")
print(f"December data shape: {december_df.shape}")

print("\n📋 Train Data (first 5 rows):")
print(train_df.head())

print("\n📋 Validation Data (first 5 rows):")
print(validation_df.head())

print("\n📋 December Data (first 5 rows):")
print(december_df.head())

print("\n✅ Data import complete! Ready for next step.")

Mounted at /content/drive
IMPORTING DATA FROM GOOGLE DRIVE

📥 Loading datasets...
✅ All files loaded successfully!

📊 DATASET SUMMARY:
------------------------------------------------------------
Train data shape: (48000, 14)
Validation data shape: (12000, 13)
Template data shape: (12000, 2)
December data shape: (31, 7)

📋 Train Data (first 5 rows):
     load_id        pickup      delivery  pickup_lat  pickup_lon  \
0  TR-000001      Richmond     Baltimore    38.09122   -76.78906   
1  TR-000002      Richmond  Philadelphia    38.09122   -76.78906   
2  TR-000003  Philadelphia     Green Bay    39.22317   -72.96710   
3  TR-000004      Hartford       Atlanta    39.55328   -72.18051   
4  TR-000005        Dallas     Nashville    31.83025   -94.38343   

   delivery_lat  delivery_lon  distance equipment   weight        date  \
0      38.16908     -72.74564     274.3   Dry Van  30658.0  2025-01-01   
1      39.22317     -72.96710     280.5    Reefer  17555.0  2025-01-01   
2      44.30296  

In [3]:
print("=" * 60)
print("STEP 1: DATA EXPLORATION & ANALYSIS")
print("=" * 60)

# 1. Check data types and missing values
print("\n1️⃣ TRAIN DATA INFO:")
print("-" * 60)
print(f"Shape: {train_df.shape}")
print(f"\nData types:")
print(train_df.dtypes)
print(f"\nMissing values:")
print(train_df.isnull().sum())

# 2. Check target variable
print("\n2️⃣ TARGET VARIABLE (posted_rate):")
print("-" * 60)
print(f"Mean: ${train_df['posted_rate'].mean():.2f}")
print(f"Median: ${train_df['posted_rate'].median():.2f}")
print(f"Min: ${train_df['posted_rate'].min():.2f}")
print(f"Max: ${train_df['posted_rate'].max():.2f}")
print(f"Std Dev: ${train_df['posted_rate'].std():.2f}")

# 3. Examine numerical columns
print("\n3️⃣ NUMERICAL FEATURES:")
print("-" * 60)
numerical_cols = train_df.select_dtypes(include=['int64', 'float64']).columns
print(f"Numerical columns: {list(numerical_cols)}")
print("\nBasic statistics:")
print(train_df[numerical_cols].describe())

# 4. Examine categorical columns
print("\n4️⃣ CATEGORICAL FEATURES:")
print("-" * 60)
categorical_cols = train_df.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_cols)}")
for col in categorical_cols:
    print(f"\n{col} - Unique values: {train_df[col].nunique()}")
    print(f"Top 5 values:\n{train_df[col].value_counts().head()}")

# 5. Check for duplicates
print("\n5️⃣ DATA QUALITY:")
print("-" * 60)
print(f"Total rows: {len(train_df)}")
print(f"Duplicate rows: {train_df.duplicated().sum()}")
print(f"Duplicate load_ids: {train_df['load_id'].duplicated().sum()}")

# 6. Date analysis
print("\n6️⃣ DATE RANGE:")
print("-" * 60)
train_df['date'] = pd.to_datetime(train_df['date'])
print(f"Date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Total days: {(train_df['date'].max() - train_df['date'].min()).days}")

print("\n" + "=" * 60)
print("✅ STEP 1 COMPLETE - DATA EXPLORATION DONE")
print("=" * 60)

STEP 1: DATA EXPLORATION & ANALYSIS

1️⃣ TRAIN DATA INFO:
------------------------------------------------------------
Shape: (48000, 14)

Data types:
load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

Missing values:
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

2️⃣ TARGET VARIABLE (posted_rate):
------------------------------------------------------------
Mean: $2373.98
Median: $2030.76
Min: $57.22
Max: $25533.00
Std Dev: $1486.49

3️⃣ NUMERICAL 

In [12]:
print("=" * 60)
print("STEP 2: FEATURE ENGINEERING & PREPROCESSING")
print("=" * 60)

# Make copies to work with
train_processed = train_df.copy()
val_processed = validation_df.copy()
dec_processed = december_df.copy()

# Convert dates
train_processed['date'] = pd.to_datetime(train_processed['date'])
val_processed['date'] = pd.to_datetime(val_processed['date'])
dec_processed['date'] = pd.to_datetime(dec_processed['date'])

# ============================================================
# 1. HANDLE MISSING VALUES
# ============================================================
print("\n1️⃣ HANDLING MISSING VALUES:")
print("-" * 60)

# Fill weight missing values with median (from training data)
weight_median = train_processed['weight'].median()
train_processed['weight'] = train_processed['weight'].fillna(weight_median)
val_processed['weight'] = val_processed['weight'].fillna(weight_median)
# For dec_processed, if 'weight' does not exist, create it and fill with median
if 'weight' not in dec_processed.columns:
    dec_processed['weight'] = weight_median
else:
    dec_processed['weight'] = dec_processed['weight'].fillna(weight_median)

# Fill market_index missing values with median
market_median = train_processed['market_index'].median()
train_processed['market_index'] = train_processed['market_index'].fillna(market_median)
val_processed['market_index'] = val_processed['market_index'].fillna(market_median)
# For dec_processed, if 'market_index' does not exist, create it and fill with median
if 'market_index' not in dec_processed.columns:
    dec_processed['market_index'] = market_median
else:
    dec_processed['market_index'] = dec_processed['market_index'].fillna(market_median)

print(f"✅ Filled weight with median: {weight_median:.2f}")
print(f"✅ Filled market_index with median: {market_median:.4f}")
print(f"✅ Remaining missing values in train: {train_processed.isnull().sum().sum()}")

# ============================================================
# 2. CREATE DATE FEATURES
# ============================================================
print("\n2️⃣ CREATING DATE FEATURES:")
print("-" * 60)

def create_date_features(df):
    df_copy = df.copy()
    df_copy['month'] = df_copy['date'].dt.month
    df_copy['day'] = df_copy['date'].dt.day
    df_copy['dayofweek'] = df_copy['date'].dt.dayofweek
    df_copy['quarter'] = df_copy['date'].dt.quarter
    df_copy['week'] = df_copy['date'].dt.isocalendar().week
    return df_copy

train_processed = create_date_features(train_processed)
val_processed = create_date_features(val_processed)
dec_processed = create_date_features(dec_processed)

print(f"✅ Added: month, day, dayofweek, quarter, week")

# ============================================================
# 3. ENCODE CATEGORICAL VARIABLES
# ============================================================
print("\n3️⃣ ENCODING CATEGORICAL VARIABLES:")
print("-" * 60)

# Frequency encoding for pickup and delivery (based on training data)
pickup_freq = train_processed['pickup'].value_counts().to_dict()
delivery_freq = train_processed['delivery'].value_counts().to_dict()

train_processed['pickup_freq'] = train_processed['pickup'].map(pickup_freq)
train_processed['delivery_freq'] = train_processed['delivery'].map(delivery_freq)

val_processed['pickup_freq'] = val_processed['pickup'].map(pickup_freq).fillna(1)
val_processed['delivery_freq'] = val_processed['delivery'].map(delivery_freq).fillna(1)

dec_processed['pickup_freq'] = dec_processed['pickup'].map(pickup_freq).fillna(1)
dec_processed['delivery_freq'] = dec_processed['delivery'].map(delivery_freq).fillna(1)

print(f"✅ Frequency encoded pickup and delivery")
print(f"   Unique pickup locations: {len(pickup_freq)}")
print(f"   Unique delivery locations: {len(delivery_freq)}")

# One-hot encode equipment
train_processed = pd.get_dummies(train_processed, columns=['equipment'], drop_first=True)
val_processed = pd.get_dummies(val_processed, columns=['equipment'], drop_first=True)
dec_processed = pd.get_dummies(dec_processed, columns=['equipment'], drop_first=True)

print(f"✅ One-hot encoded equipment type")
equipment_cols = [col for col in train_processed.columns if col.startswith('equipment_')]
print(f"   Equipment columns: {equipment_cols}")

# ============================================================
# 4. REMOVE UNNECESSARY COLUMNS
# ============================================================
print("\n4️⃣ REMOVING UNNECESSARY COLUMNS:")
print("-" * 60)

drop_cols = ['load_id', 'date', 'pickup', 'delivery']
train_processed.drop(columns=drop_cols, inplace=True, errors='ignore')
val_processed.drop(columns=drop_cols, inplace=True, errors='ignore')
dec_processed.drop(columns=drop_cols, inplace=True, errors='ignore')

print(f"✅ Dropped: {drop_cols}")

# ============================================================
# 5. ALIGN FEATURES ACROSS DATASETS
# ============================================================
print("\n5️⃣ ALIGNING FEATURES ACROSS DATASETS:")
print("-" * 60)

# Ensure all datasets have same columns based on train_processed's initial columns
# Crucially, if posted_rate exists in train_processed, it will be added to others here.
train_processed, val_processed = train_processed.align(val_processed, join='left', axis=1, fill_value=0)
train_processed, dec_processed = train_processed.align(dec_processed, join='left', axis=1, fill_value=0)

# Fill any remaining NaNs with 0
train_processed = train_processed.fillna(0)
val_processed = val_processed.fillna(0)
dec_processed = dec_processed.fillna(0)

print(f"✅ Train shape: {train_processed.shape}")
print(f"✅ Validation shape: {val_processed.shape}")
print(f"✅ December shape: {dec_processed.shape}")

# ============================================================
# 6. SEPARATE FEATURES AND TARGET
# ============================================================
print("\n6️⃣ SEPARATING FEATURES AND TARGET:")
print("-" * 60)

# Extract target variable
y_train = train_df['posted_rate']

# Get features for training (all columns except target)
X_train = train_processed.drop(columns=['posted_rate'], errors='ignore')

# For validation and December, ensure they have the same columns as X_train
# First, drop 'posted_rate' from them if it was included during alignment
val_processed_features = val_processed.drop(columns=['posted_rate'], errors='ignore')
dec_processed_features = dec_processed.drop(columns=['posted_rate'], errors='ignore')

# Then, reindex to explicitly match X_train's columns (adding missing columns as 0 if any)
X_val = val_processed_features.reindex(columns=X_train.columns, fill_value=0)
X_dec = dec_processed_features.reindex(columns=X_train.columns, fill_value=0)


print(f"✅ Features: {X_train.shape[1]} columns")
print(f"✅ Training target (y_train): {y_train.shape[0]} samples")
print(f"\nFeature columns:")
print(list(X_train.columns))

# ============================================================
# 7. SUMMARY STATISTICS
# ============================================================
print("\n7️⃣ FEATURE SUMMARY:")
print("-" * 60)
print(f"Total training samples: {X_train.shape[0]}")
print(f"Total features: {X_train.shape[1]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"December samples: {X_dec.shape[0]}")
print(f"\nData types in processed data:")
print(X_train.dtypes.value_counts())

print("\n" + "=" * 60)
print("✅ STEP 2 COMPLETE - DATA READY FOR MODELING")
print("=" * 60)

STEP 2: FEATURE ENGINEERING & PREPROCESSING

1️⃣ HANDLING MISSING VALUES:
------------------------------------------------------------
✅ Filled weight with median: 31436.50
✅ Filled market_index with median: 1.0558
✅ Remaining missing values in train: 0

2️⃣ CREATING DATE FEATURES:
------------------------------------------------------------
✅ Added: month, day, dayofweek, quarter, week

3️⃣ ENCODING CATEGORICAL VARIABLES:
------------------------------------------------------------
✅ Frequency encoded pickup and delivery
   Unique pickup locations: 64
   Unique delivery locations: 64
✅ One-hot encoded equipment type
   Equipment columns: ['equipment_Flatbed', 'equipment_Reefer']

4️⃣ REMOVING UNNECESSARY COLUMNS:
------------------------------------------------------------
✅ Dropped: ['load_id', 'date', 'pickup', 'delivery']

5️⃣ ALIGNING FEATURES ACROSS DATASETS:
------------------------------------------------------------
✅ Train shape: (48000, 18)
✅ Validation shape: (12000, 18)
✅ 

## STEP 3: MODEL TRAINING AND EVALUATION

In this step, we will train four different regression models and evaluate their performance on the validation set. We'll also make predictions for the December dataset.

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("============================================================")
print("STEP 3: DATA PREPARATION FOR MODELING")
print("============================================================")

print("1. Loading datasets...")
# Correct path to Google Drive data
data_path = '/content/drive/My Drive/Data'

train_df = pd.read_csv(f'{data_path}/train-test.csv')
validation_df = pd.read_csv(f'{data_path}/validation.csv')
template_df = pd.read_csv(f'{data_path}/validation-predictions-template.csv')
december_df = pd.read_csv(f'{data_path}/december-chart-inputs (1).csv')

# Create copies for storing predictions later, these will be updated by each model
val_processed = validation_df.copy()
dec_processed = december_df.copy()

# Configure mapping from historical 'posted_rate' to evaluation 'predicted_rate'
target_col = 'posted_rate'

print("2. Processing features...")
def process_features(df_input):
    processed = df_input.copy()

    # Extract numerical date parts
    processed['date'] = pd.to_datetime(processed['date'])
    processed['month'] = processed['date'].dt.month
    processed['day'] = processed['date'].dt.day
    processed['dayofweek'] = processed['date'].dt.dayofweek

    # Handle missing weight/market_index using median from the original train_df
    if 'weight' in processed.columns:
        weight_median = train_df['weight'].median() # Use train_df's median
        processed['weight'] = processed['weight'].fillna(weight_median)
    if 'market_index' in processed.columns:
        market_median = train_df['market_index'].median() # Use train_df's median
        processed['market_index'] = processed['market_index'].fillna(market_median)

    # Handle text columns using frequency encoding maps from training data
    for col in ['pickup', 'delivery']:
        freq_map = train_df[col].value_counts().to_dict()
        processed[f'{col}_freq'] = processed[col].map(freq_map).fillna(1)

    # Convert equipment categories to dummy columns
    processed = pd.get_dummies(processed, columns=['equipment'], drop_first=True)

    # Keep only pure numeric columns for training
    exclude = ['load_id', 'date', 'pickup', 'delivery', target_col, 'predicted_rate']
    features = [c for c in processed.columns if c not in exclude]
    return processed[features]

# Generate feature matrices
X_train_full_raw = process_features(train_df)
y_train_full = train_df[target_col]
X_val_raw = process_features(validation_df)
X_dec_raw = process_features(december_df)

# Align structural dimensions across subsets to prevent mismatch bugs
# Create a union of all columns present in any of the raw feature sets
all_features_union = X_train_full_raw.columns.union(X_val_raw.columns).union(X_dec_raw.columns)

# Reindex all feature sets to this common set of columns
X_train_full = X_train_full_raw.reindex(columns=all_features_union, fill_value=0)
X_val = X_val_raw.reindex(columns=all_features_union, fill_value=0)
X_dec = X_dec_raw.reindex(columns=all_features_union, fill_value=0)

# Ensure consistent data types across aligned dataframes
# This step helps prevent potential issues with some models expecting specific dtypes
for col in X_train_full.columns:
    if col in X_val.columns and X_val[col].dtype != X_train_full[col].dtype:
        X_val[col] = X_val[col].astype(X_train_full[col].dtype)
    if col in X_dec.columns and X_dec[col].dtype != X_train_full[col].dtype:
        X_dec[col] = X_dec[col].astype(X_train_full[col].dtype)

print("3. Splitting data for training and testing...")
X_train, X_test, y_train, y_test = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

print("============================================================")
print("✅ Data preparation complete! Ready for model training.")
print("============================================================")

STEP 3: DATA PREPARATION FOR MODELING
1. Loading datasets...
2. Processing features...
3. Splitting data for training and testing...
✅ Data preparation complete! Ready for model training.


## STEP 4: MODEL TRAINING & PREDICTION

### Model 1: Random Forest Regressor

In [26]:
print("============================================================")
print("MODEL 1: RANDOM FOREST REGRESSOR")
print("============================================================")

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the model
# Multi-core processing enabled (-1)
rf_model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred_rf = rf_model.predict(X_val)
val_processed['predicted_rate_rf'] = y_val_pred_rf # Store predictions for validation set

# Evaluate the model (cannot be done directly on validation_df as 'posted_rate' is missing)
print("⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.")

print(f"✅ Random Forest Regressor - Predictions for validation set generated.")

# Make predictions on December data
dec_processed['predicted_rate_rf'] = rf_model.predict(X_dec) # Store in dec_processed
print("✅ Predictions for December data generated.")

print(f"-> Train accuracy (R²): {rf_model.score(X_train, y_train):.4f}")
print(f"-> Test validation accuracy (R²): {rf_model.score(X_test, y_test):.4f}")


MODEL 1: RANDOM FOREST REGRESSOR
⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.
✅ Random Forest Regressor - Predictions for validation set generated.
✅ Predictions for December data generated.
-> Train accuracy (R²): 0.9567
-> Test validation accuracy (R²): 0.8372


### Model 2: Linear Regression

In [27]:
print("============================================================")
print("MODEL 2: LINEAR REGRESSION")
print("============================================================")

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred_lr = linear_model.predict(X_val)
val_processed['predicted_rate_lr'] = y_val_pred_lr # Store predictions for validation set

# Evaluate the model (cannot be done directly on validation_df as 'posted_rate' is missing)
print("⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.")

print(f"✅ Linear Regression - Predictions for validation set generated.")

# Make predictions on December data
dec_processed['predicted_rate_lr'] = linear_model.predict(X_dec)
print("✅ Predictions for December data generated.")

MODEL 2: LINEAR REGRESSION
⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.
✅ Linear Regression - Predictions for validation set generated.
✅ Predictions for December data generated.


### Model 3: XGBoost Regressor

In [28]:
print("============================================================")
print("MODEL 3: XGBOOST REGRESSOR")
print("============================================================")

import xgboost as xgb

# Initialize and train the model
# Using some default parameters, these can be tuned later
xgb_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)
xgb_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred_xgb = xgb_model.predict(X_val)
val_processed['predicted_rate_xgb'] = y_val_pred_xgb # Store predictions for validation set

# Evaluate the model (cannot be done directly on validation_df as 'posted_rate' is missing)
print("⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.")

print(f"✅ XGBoost Regressor - Predictions for validation set generated.")

# Make predictions on December data
dec_processed['predicted_rate_xgb'] = xgb_model.predict(X_dec) # Store in dec_processed
print("✅ Predictions for December data generated.")

MODEL 3: XGBOOST REGRESSOR
⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.
✅ XGBoost Regressor - Predictions for validation set generated.
✅ Predictions for December data generated.


### Model 4: CatBoost Regressor

In [29]:
print("============================================================")
print("MODEL 4: CATBOOST REGRESSOR")
print("============================================================")

!pip install catboost --quiet
from catboost import CatBoostRegressor

# Initialize and train the model
# CatBoost handles categorical features automatically, but we've already one-hot encoded 'equipment'
cat_model = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=0 # Suppress verbose output during training
)
cat_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred_cat = cat_model.predict(X_val)
val_processed['predicted_rate_cat'] = y_val_pred_cat # Store predictions for validation set

# Evaluate the model (cannot be done directly on validation_df as 'posted_rate' is missing)
print("⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.")

print(f"✅ CatBoost Regressor - Predictions for validation set generated.")

# Make predictions on December data
dec_processed['predicted_rate_cat'] = cat_model.predict(X_dec) # Store in dec_processed
print("✅ Predictions for December data generated.")

MODEL 4: CATBOOST REGRESSOR
⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.
✅ CatBoost Regressor - Predictions for validation set generated.
✅ Predictions for December data generated.


### Model 5: Gradient Boosting Regressor

In [30]:
print("============================================================")
print("MODEL 5: GRADIENT BOOSTING REGRESSOR")
print("============================================================")

from sklearn.ensemble import GradientBoostingRegressor

# Initialize and train the model
# Using some default parameters, these can be tuned later
gbr_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gbr_model.fit(X_train, y_train)

# Make predictions on validation set
y_val_pred_gbr = gbr_model.predict(X_val)
val_processed['predicted_rate_gbr'] = y_val_pred_gbr # Store predictions for validation set

# Evaluate the model (cannot be done directly on validation_df as 'posted_rate' is missing)
print("⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.")

print(f"✅ Gradient Boosting Regressor - Predictions for validation set generated.")

# Make predictions on December data
dec_processed['predicted_rate_gbr'] = gbr_model.predict(X_dec) # Store in dec_processed
print("✅ Predictions for December data generated.")

MODEL 5: GRADIENT BOOSTING REGRESSOR
⚠️ Cannot calculate MSE/R2 for validation_df as 'posted_rate' column is missing in validation_df.
✅ Gradient Boosting Regressor - Predictions for validation set generated.
✅ Predictions for December data generated.


## STEP 5: PREDICTION SUMMARY

In [31]:
print("\n" + "=" * 60)
print("SUMMARY OF DECEMBER PREDICTIONS")
print("=" * 60)

# Displaying predictions for December from Random Forest
print("\nRandom Forest December Predictions (first 5 rows):")
print(dec_processed[['predicted_rate_rf']].head())

# Displaying predictions for December from Linear Regression
print("\nLinear Regression December Predictions (first 5 rows):")
print(dec_processed[['predicted_rate_lr']].head())

# Displaying predictions for December from XGBoost
print("\nXGBoost December Predictions (first 5 rows):")
print(dec_processed[['predicted_rate_xgb']].head())

print("\nCatBoost December Predictions (first 5 rows):")
print(dec_processed[['predicted_rate_cat']].head())

print("\nGradient Boosting December Predictions (first 5 rows):")
print(dec_processed[['predicted_rate_gbr']].head())

print("\n" + "=" * 60)
print("✅ ALL MODEL TRAINING & PREDICTION COMPLETE")
print("============================================================")

# Optional: Displaying the full dec_processed DataFrame with all predictions
# print("\nFull Dec Processed DataFrame with all predictions (first 5 rows):")
# print(dec_processed.head())

# Optional: Save final dec_processed with all predictions to a CSV
# dec_processed.to_csv(f'{data_path}/december_chart_inputs_with_all_predictions.csv', index=False)


SUMMARY OF DECEMBER PREDICTIONS

Random Forest December Predictions (first 5 rows):
   predicted_rate_rf
0        1695.674786
1        1683.624454
2        1683.624454
3        1683.624454
4        1690.112379

Linear Regression December Predictions (first 5 rows):
   predicted_rate_lr
0        1129.266151
1        1130.186757
2        1131.107362
3        1132.027968
4        1132.948573

XGBoost December Predictions (first 5 rows):
   predicted_rate_xgb
0         2945.338623
1         2981.822021
2         1050.717651
3         1050.717651
4         1053.943481

CatBoost December Predictions (first 5 rows):
   predicted_rate_cat
0          871.219267
1          878.753667
2          894.505155
3          877.999323
4          877.004946

Gradient Boosting December Predictions (first 5 rows):
   predicted_rate_gbr
0          925.023997
1          925.023997
2          925.023997
3          925.023997
4          921.910111

✅ ALL MODEL TRAINING & PREDICTION COMPLETE


## STEP 6: GENERATE FINAL SUBMISSION FILE

Based on your task, you need to create a `validation_predictions.csv` file by filling the `predicted_rate` column in `validation-predictions-template.csv` with predictions from one of your models. Let's use the Random Forest Regressor's predictions for this example, but you could choose any of the models.


In [32]:
# Make a copy of the template DataFrame
submission_df = template_df.copy()

# Choose which model's predictions to use for the submission
# For example, using Random Forest Regressor predictions:
submission_df['predicted_rate'] = val_processed['predicted_rate_rf']

# Display the first few rows of the submission file
print("First 5 rows of the validation_predictions.csv:")
display(submission_df.head())

# Save the completed file as validation_predictions.csv
submission_filename = 'validation_predictions.csv'
submission_df.to_csv(submission_filename, index=False)

print(f"\n✅ Successfully created {submission_filename} for submission!")


First 5 rows of the validation_predictions.csv:


,load_id,predicted_rate
0,TE-000001,957.005814
1,TE-000002,4884.126655
2,TE-000003,4990.911039
3,TE-000004,4447.007280
4,TE-000005,1962.918689



✅ Successfully created validation_predictions.csv for submission!


## STEP 7: MODEL SELECTION AND EVALUATION ON TEST SET

To decide which model's predictions to use for the final submission and to understand their relative performance, let's evaluate all models on the `X_test` dataset from your training data split.

In [33]:
from sklearn.metrics import mean_squared_error, r2_score

model_performance = {}

# Evaluate Random Forest Regressor
y_test_pred_rf = rf_model.predict(X_test)
model_performance['Random Forest'] = {
    'R2': r2_score(y_test, y_test_pred_rf),
    'MSE': mean_squared_error(y_test, y_test_pred_rf)
}

# Evaluate Linear Regression
y_test_pred_lr = linear_model.predict(X_test)
model_performance['Linear Regression'] = {
    'R2': r2_score(y_test, y_test_pred_lr),
    'MSE': mean_squared_error(y_test, y_test_pred_lr)
}

# Evaluate XGBoost Regressor
y_test_pred_xgb = xgb_model.predict(X_test)
model_performance['XGBoost'] = {
    'R2': r2_score(y_test, y_test_pred_xgb),
    'MSE': mean_squared_error(y_test, y_test_pred_xgb)
}

# Evaluate CatBoost Regressor
y_test_pred_cat = cat_model.predict(X_test)
model_performance['CatBoost'] = {
    'R2': r2_score(y_test, y_test_pred_cat),
    'MSE': mean_squared_error(y_test, y_test_pred_cat)
}

# Evaluate Gradient Boosting Regressor
y_test_pred_gbr = gbr_model.predict(X_test)
model_performance['Gradient Boosting'] = {
    'R2': r2_score(y_test, y_test_pred_gbr),
    'MSE': mean_squared_error(y_test, y_test_pred_gbr)
}

# Display performance
performance_df = pd.DataFrame.from_dict(model_performance, orient='index')
performance_df = performance_df.sort_values(by='R2', ascending=False)

print("\n--- Model Performance on Test Set (X_test, y_test) ---")
display(performance_df)

print("\nThe model with the highest R-squared score and lowest Mean Squared Error on the test set is generally considered the best performing model for this dataset.")
print("Based on these results, you can choose which model's predictions to use for your final `validation_predictions.csv` submission.")



--- Model Performance on Test Set (X_test, y_test) ---


,R2,MSE
CatBoost,0.856666,317482.338108
Gradient Boosting,0.855791,319418.378552
Linear Regression,0.852259,327243.158938
XGBoost,0.848560,335435.364960
Random Forest,0.837174,360655.069006



The model with the highest R-squared score and lowest Mean Squared Error on the test set is generally considered the best performing model for this dataset.
Based on these results, you can choose which model's predictions to use for your final `validation_predictions.csv` submission.


## STEP 8: PREPARE FILES FOR `score.py` AND RUN SCORING SCRIPT

We will now prepare the two `.csv` files required by `score.py` and then execute the script. We will use the predictions from the **CatBoost Regressor**, as it demonstrated the best performance on the test set.

### 8.1 Update `validation_predictions.csv` with CatBoost Predictions

In [34]:
# Make a copy of the template DataFrame
submission_df_catboost = template_df.copy()

# Use CatBoost Regressor predictions for the submission
submission_df_catboost['predicted_rate'] = val_processed['predicted_rate_cat']

# Display the first few rows of the updated submission file
print("First 5 rows of the validation_predictions.csv (CatBoost):")
display(submission_df_catboost.head())

# Save the completed file as validation_predictions.csv
submission_filename = 'validation_predictions.csv'
submission_df_catboost.to_csv(submission_filename, index=False)

print(f"\n✅ Successfully updated {submission_filename} with CatBoost predictions!")


First 5 rows of the validation_predictions.csv (CatBoost):


,load_id,predicted_rate
0,TE-000001,863.698284
1,TE-000002,5046.191105
2,TE-000003,5255.550738
3,TE-000004,4114.980496
4,TE-000005,1836.791472



✅ Successfully updated validation_predictions.csv with CatBoost predictions!


### 8.2 Create `december_predictions.csv` for Chart Generation

The `score.py` script expects the December predictions in a specific format: the original December columns plus the `predicted_rate`. We will create this file using CatBoost's predictions.

In [35]:
# Start with the original december_df
december_predictions_for_chart = december_df.copy()

# Fill the 'predicted_rate' column with CatBoost's predictions
december_predictions_for_chart['predicted_rate'] = dec_processed['predicted_rate_cat']

# Ensure the columns are in the order expected by score.py
expected_december_cols = ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']
december_predictions_for_chart = december_predictions_for_chart[expected_december_cols]

# Display the first few rows
print("First 5 rows of the december_predictions.csv (CatBoost):")
display(december_predictions_for_chart.head())

# Save the file
december_predictions_filename = 'december_predictions.csv'
december_predictions_for_chart.to_csv(december_predictions_filename, index=False)

print(f"\n✅ Successfully created {december_predictions_filename} with CatBoost predictions!")


First 5 rows of the december_predictions.csv (CatBoost):


,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,871.219267
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,878.753667
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,894.505155
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,877.999323
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,877.004946



✅ Successfully created december_predictions.csv with CatBoost predictions!


### 8.3 Save `score.py` and Run It

Now we will save the `score.py` content to a file and execute it with the prepared CSVs. The script will validate your submission files and generate `candidate_december.png`.

In [38]:
# Generate requirements.txt based on imported libraries
import pkg_resources

# List of libraries used in the notebook (add any others you might have used manually)
installed_packages = pkg_resources.working_set
used_libraries = [
    'pandas',
    'numpy',
    'scikit-learn',
    'xgboost',
    'catboost',
    'matplotlib'
]

requirements = []
for package in installed_packages:
    if package.key in used_libraries:
        requirements.append(f"{package.key}=={package.version}")

requirements_content = "\n".join(sorted(requirements))

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print(f"✅ `requirements.txt` created with content:\n---\n{requirements_content}\n---")


✅ `requirements.txt` created with content:
---
catboost==1.2.10
matplotlib==3.10.0
numpy==2.0.2
pandas==2.2.2
scikit-learn==1.6.1
xgboost==3.3.0
---


/tmp/ipykernel_765/3077909065.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


A `README.md` file is crucial for a GitHub repository. Here's a basic structure that you can expand upon in your report:

In [39]:
# Create a basic README.md file
readme_content = '''
# Freight Pricing Prediction Project

This repository contains the solution for predicting freight prices based on historical data and various logistical factors.

## Project Overview

The goal of this project is to develop a machine learning model to accurately predict freight rates for various loads, optimizing pricing strategies. The solution involves data loading, comprehensive preprocessing, feature engineering, training and evaluation of multiple regression models, and generating submission files.

## Dataset

The project utilizes several datasets provided:
- `train-test.csv`: Labeled development data for model training and testing.
- `validation.csv`: Unlabeled data for final predictions.
- `validation-predictions-template.csv`: Template for the final submission file.
- `december-chart-inputs (1).csv`: Data used to generate a December rate prediction chart.

## Solution Approach

1.  **Data Loading & Initial Exploration:** Datasets are loaded and basic exploratory data analysis is performed to understand distributions, missing values, and data types.
2.  **Feature Engineering & Preprocessing:**
    -   Handling missing values (e.g., using median imputation for `weight` and `market_index`).
    -   Extracting time-based features from the `date` column (month, day, day of week, quarter, week).
    -   Encoding categorical variables using frequency encoding for `pickup`/`delivery` locations and one-hot encoding for `equipment`.
    -   Aligning features across training, validation, and December datasets to ensure consistency.
3.  **Model Training & Evaluation:**
    -   The training data (`train-test.csv`) is split into training and test sets.
    -   Multiple regression models are trained: Random Forest Regressor, Linear Regression, XGBoost Regressor, CatBoost Regressor, and Gradient Boosting Regressor.
    -   Models are evaluated on the internal test set using R-squared and Mean Squared Error metrics.
4.  **Prediction Generation:**
    -   Predictions are generated for the `validation.csv` and `december-chart-inputs (1).csv` datasets using the best-performing model (CatBoost Regressor).
5.  **Submission File Creation & Scoring:**
    -   A `validation_predictions.csv` file is generated following the specified format.
    -   A `december_predictions.csv` file is prepared for chart generation.
    -   The `score.py` script is used to validate the output files and generate a visualization of December predictions.

## Repository Structure

-   `[Your_Colab_Notebook_Name].ipynb`: The main notebook containing all the code for data processing, modeling, and evaluation.
-   `requirements.txt`: Lists all Python package dependencies.
-   `validation_predictions.csv`: The final submission file containing predicted rates for the validation set.
-   `december_predictions.csv`: Predictions for December used by the scoring script.
-   `scorer_results/candidate_december.png`: Visualization of the December predictions generated by `score.py`.
-   `score.py`: The provided scoring script.

## How to Run

1.  **Environment Setup:** Ensure you have Python 3.x installed. Install the required libraries using `pip install -r requirements.txt`.
2.  **Data:** Place the provided `.csv` files (`train-test.csv`, `validation.csv`, `validation-predictions-template.csv`, `december-chart-inputs (1).csv`) in the specified `data_path` (e.g., `/content/drive/My Drive/Data` if running in Google Colab).
3.  **Execute Notebook:** Run all cells in the Jupyter/Colab notebook sequentially.
4.  **Review Outputs:** Check the generated `validation_predictions.csv`, `december_predictions.csv`, and the `scorer_results/candidate_december.png`.

## Author

Abdul Saboor, Machine learning engineer / Software engineer
'''

with open('README.md', 'w') as f:
    f.write(readme_content)

print("✅ `README.md` created successfully!")


✅ `README.md` created successfully!


In [36]:
# Save the score.py content to a file
score_py_content = '''
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


EXPECTED_ROWS = 12_000
EXPECTED_IDS = {f"TE-{index:06d}" for index in range(1, EXPECTED_ROWS + 1)}
DECEMBER_DATES = pd.date_range("2025-12-01", "2025-12-31", freq="D")
FIXED_PICKUP = "Lexington"
FIXED_DELIVERY = "Fort Wayne"
FIXED_DISTANCE = 360.0
FIXED_EQUIPMENT = "Dry Van"
FIXED_WEIGHT = 32_000.0


def fail(message: str) -> None:
    raise SystemExit(f"ERROR: {message}")


def read_csv(path: Path, label: str) -> pd.DataFrame:
    if not path.is_file():
        fail(f"{label} file not found: {path}")
    try:
        return pd.read_csv(path)
    except Exception as exc:
        fail(f"could not read {label}: {exc}")


def numeric_series(frame: pd.DataFrame, column: str, label: str) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    if values.isna().any() or not np.isfinite(values).all():
        fail(f"{label} contains invalid {column} values")
    return values.astype(float)


def validate_predictions(predictions: pd.DataFrame) -> None:
    if list(predictions.columns) != ["load_id", "predicted_rate"]:
        fail("predictions must contain exactly two columns in this order: load_id,predicted_rate")
    if len(predictions) != EXPECTED_ROWS:
        fail(f"predictions must contain exactly {EXPECTED_ROWS:,} rows")
    if predictions["load_id"].isna().any() or predictions["load_id"].duplicated().any():
        fail("predictions contains missing or duplicate load_id values")

    submitted_ids = set(predictions["load_id"].astype(str))
    missing = EXPECTED_IDS - submitted_ids
    extra = submitted_ids - EXPECTED_IDS
    if missing or extra:
        fail(
            "prediction IDs do not match the validation set "
            f"(missing={len(missing)}, extra={len(extra)})"
        )

    predicted_rate = numeric_series(predictions, "predicted_rate", "predictions")
    if (predicted_rate <= 0).any():
        fail("predictions contains non-positive predicted_rate values")


def validate_december(frame: pd.DataFrame) -> pd.DataFrame:
    columns = ["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]
    if list(frame.columns) != columns:
        fail("December predictions must keep the original seven columns and column order")

    result = frame.copy()
    result["date"] = pd.to_datetime(result["date"], errors="coerce")
    if result["date"].isna().any():
        fail("December predictions contains invalid dates")
    result["distance"] = numeric_series(result, "distance", "December predictions")
    result["weight"] = numeric_series(result, "weight", "December predictions")
    result["predicted_rate"] = numeric_series(result, "predicted_rate", "December predictions")

    if result["date"].duplicated().any():
        fail("December predictions contains duplicate dates")
    if len(result) != 31 or set(result["date"]) != set(DECEMBER_DATES):
        fail("December predictions must contain one row for every day from 2025-12-01 to 2025-12-31")
    if not result["pickup"].eq(FIXED_PICKUP).all():
        fail(f"December pickup must be {FIXED_PICKUP} for all rows")
    if not result["delivery"].eq(FIXED_DELIVERY).all():
        fail(f"December delivery must be {FIXED_DELIVERY} for all rows")
    if not np.isclose(result["distance"], FIXED_DISTANCE).all():
        fail(f"December distance must be {FIXED_DISTANCE:g} for all rows")
    if not result["equipment"].eq(FIXED_EQUIPMENT).all():
        fail(f"December equipment must be {FIXED_EQUIPMENT} for all rows")
    if not np.isclose(result["weight"], FIXED_WEIGHT).all():
        fail(f"December weight must be {FIXED_WEIGHT:g} for all rows")
    if (result["predicted_rate"] <= 0).any():
        fail("December predicted_rate values must be positive")
    return result.sort_values("date")


def save_december_chart(december: pd.DataFrame, output: Path) -> None:
    figure, axis = plt.subplots(figsize=(10.8, 4.8), dpi=180)
    color = "#064A56"
    axis.plot(
        december["date"],
        december["predicted_rate"],
        color=color,
        linewidth=2.6,
        marker="o",
        markersize=3.2,
    )
    floor = float(december["predicted_rate"].min())
    axis.fill_between(
        december["date"],
        december["predicted_rate"],
        floor - max(10.0, floor * 0.02),
        color=color,
        alpha=0.08,
    )
    axis.set_title("Candidate: December 2025 Predicted Load Rate", loc="left", fontsize=15, fontweight="bold", pad=12)
    axis.set_ylabel("Predicted rate ($)")
    axis.grid(axis="y", color="#D9E2E4", linewidth=0.8)
    axis.spines[["top", "right"]].set_visible(False)
    axis.spines[["left", "bottom"]].set_color("#9DAFB3")
    axis.tick_params(axis="x", rotation=35)
    axis.text(
        0,
        -0.40,
        "Fixed inputs: Lexington to Fort Wayne | 360 miles | Dry Van | 32,000 lb | only date changes",
        transform=axis.transAxes,
        fontsize=9.5,
        color="#455A60",
    )
    figure.tight_layout(rect=(0, 0.12, 1, 1))
    figure.savefig(output, bbox_inches="tight")
    plt.close(figure)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Validate candidate output files and generate the fixed December chart."
    )
    parser.add_argument("--predictions", required=True, help="CSV with load_id,predicted_rate")
    parser.add_argument(
        "--december-predictions",
        required=True,
        help="Completed data/december_chart_inputs.csv",
    )
    parser.add_argument("--output-dir", default="scorer_results")
    args = parser.parse_args()

    validate_predictions(read_csv(Path(args.predictions), "predictions"))
    december = validate_december(read_csv(Path(args.december_predictions), "December predictions"))
    output = Path(args.output_dir)
    output.mkdir(parents=True, exist_ok=True)
    chart = output / "candidate_december.png"
    save_december_chart(december, chart)

    print(f"Validated {EXPECTED_ROWS:,} final predictions.")
    print("Validated 31 fixed December predictions.")
    print(f"Created chart: {chart}")
    print("Final validation metrics are calculated by Spotter after submission.")


if __name__ == "__main__":
    main()
'''

with open('score.py', 'w') as f:
    f.write(score_py_content)

print("✅ `score.py` saved successfully!")

# Run the score.py script
!python score.py --predictions validation_predictions.csv --december-predictions december_predictions.csv

print("\n✅ `score.py` executed successfully!")
print("You should now find `candidate_december.png` in the `scorer_results` directory.")


✅ `score.py` saved successfully!
Validated 12,000 final predictions.
Validated 31 fixed December predictions.
Created chart: scorer_results/candidate_december.png
Final validation metrics are calculated by Spotter after submission.

✅ `score.py` executed successfully!
You should now find `candidate_december.png` in the `scorer_results` directory.


In [37]:
import shutil
import os

# Define the output directory in Google Drive
drive_output_path = os.path.join(data_path, 'scorer_results_output')
os.makedirs(drive_output_path, exist_ok=True)

# Define paths to the generated files
validation_predictions_file = 'validation_predictions.csv'
december_predictions_file = 'december_predictions.csv'
candidate_december_chart = 'scorer_results/candidate_december.png'

# Copy files to Google Drive
shutil.copy(validation_predictions_file, drive_output_path)
shutil.copy(december_predictions_file, drive_output_path)
shutil.copy(candidate_december_chart, drive_output_path)

# Provide direct download links for convenience
print(f"\n✅ Files saved to Google Drive: {drive_output_path}")
print("You can also download them directly using these links:")
print(f"* [Download validation_predictions.csv](file/{validation_predictions_file})")
print(f"* [Download december_predictions.csv](file/{december_predictions_file})")
print(f"* [Download candidate_december.png](file/{candidate_december_chart})")



✅ Files saved to Google Drive: /content/drive/My Drive/Data/scorer_results_output
You can also download them directly using these links:
* [Download validation_predictions.csv](file/validation_predictions.csv)
* [Download december_predictions.csv](file/december_predictions.csv)
* [Download candidate_december.png](file/scorer_results/candidate_december.png)


In [40]:
import shutil
import os

# Define the output directory in Google Drive
drive_output_path = os.path.join(data_path, 'scorer_results_output')
os.makedirs(drive_output_path, exist_ok=True)

# Define paths to the generated files
validation_predictions_file = 'validation_predictions.csv'
december_predictions_file = 'december_predictions.csv'
candidate_december_chart = 'scorer_results/candidate_december.png'
requirements_file = 'requirements.txt'
readme_file = 'README.md'

# Copy files to Google Drive
shutil.copy(validation_predictions_file, drive_output_path)
shutil.copy(december_predictions_file, drive_output_path)
shutil.copy(candidate_december_chart, drive_output_path)
shutil.copy(requirements_file, drive_output_path)
shutil.copy(readme_file, drive_output_path)

# Provide direct download links for convenience
print(f"\n✅ All requested files saved to Google Drive: {drive_output_path}")
print("You can also download them directly using these links:")
print(f"* [Download validation_predictions.csv](file/{validation_predictions_file})")
print(f"* [Download december_predictions.csv](file/{december_predictions_file})")
print(f"* [Download candidate_december.png](file/{candidate_december_chart})")
print(f"* [Download requirements.txt](file/{requirements_file})")
print(f"* [Download README.md](file/{readme_file})")



✅ All requested files saved to Google Drive: /content/drive/My Drive/Data/scorer_results_output
You can also download them directly using these links:
* [Download validation_predictions.csv](file/validation_predictions.csv)
* [Download december_predictions.csv](file/december_predictions.csv)
* [Download candidate_december.png](file/scorer_results/candidate_december.png)
* [Download requirements.txt](file/requirements.txt)
* [Download README.md](file/README.md)
